# 📈 TikTok Engagement Pulse: Time Series Decomposition & Stationarity Audit
**Objective:** Apply the Discovery-to-Action (DTA) framework to evaluate temporal TikTok view patterns, isolate underlying seasonality components, conduct an Augmented Dickey-Fuller (ADF) stationarity audit, and establish an optimized content distribution schedule.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Setting plot configurations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Look for an external time series file or fall back to an engineered temporal tracking frame
target_file = None
for file in os.listdir('.'):
    if file.endswith('.csv') and 'series' in file.lower():
        target_file = file
        break

if target_file:
    print(f"Ingesting time series tracking profile from: {target_file}")
    df = pd.read_csv(target_file)
    # Automatically identify date columns
    date_col = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()][0]
    val_col = [col for col in df.columns if 'view' in col.lower() or 'count' in col.lower() or 'engagement' in col.lower()][0]
    df = df.rename(columns={date_col: 'date', val_col: 'view_count'})
else:
    print("No explicit time series file isolated. Initializing standard production simulation asset...")
    # Generate a realistic 180-day daily series reflecting actual TikTok consumption habits
    np.random.seed(42)
    dates = pd.date_range(start="2026-01-01", periods=180, freq="D")

    # Construct base log-linear trend + strong weekly seasonality (peaks on Thursdays and Fridays)
    trend = np.linspace(50000, 120000, 180)
    weekly_pattern = np.array([0.8, 0.85, 0.9, 1.25, 1.3, 1.1, 0.85]) # Mon-Sun scaling weights
    seasonality = np.array([weekly_pattern[d.weekday()] for d in dates]) * 15000
    noise = np.random.normal(0, 8000, 180)

    views = trend + seasonality + noise
    df = pd.DataFrame({'date': dates, 'view_count': views})

print(f"Dataframe Sample Initialized. Row Count: {len(df)}")

In [ ]:
# 1. Standardize and cast date indices
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# 2. Resample to a strict daily ('D') frequency, filling any missing intervals via linear interpolation
df_resampled = df['view_count'].resample('D').mean().to_frame()
missing_imputed = df_resampled['view_count'].isnull().sum()
if missing_imputed > 0:
    print(f"Identified {missing_imputed} missing timestamps. Executing linear trend interpolation...")
    df_resampled['view_count'] = df_resampled['view_count'].interpolate(method='linear')

print("Time Series index successfully aligned to regularized daily interval frequency.")

In [ ]:
# Apply additive classical decomposition (Period=7 representing a standard weekly operation loop)
decomposition = seasonal_decompose(df_resampled['view_count'], model='additive', period=7)

# Extract individual structural components
trend_component = decomposition.trend
seasonal_component = decomposition.seasonal
residual_component = decomposition.resid

# Generate high-resolution structural plot layout
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

df_resampled['view_count'].plot(ax=axes[0], color='black', alpha=0.8, title='Observed TikTok Engagement Series')
axes[0].set_ylabel('Views')

trend_component.plot(ax=axes[1], color='blue', title='Isolated Structural Long-Term Trend')
axes[1].set_ylabel('Trend')

seasonal_component.plot(ax=axes[2], color='green', title='Isolated Cyclical Weekly Seasonality')
axes[2].set_ylabel('Seasonal Adjustment')

residual_component.plot(ax=axes[3], color='red', style='.', title='Stochastic Residuals (Noise)')
axes[3].set_ylabel('Irregularities')

plt.tight_layout()
plt.savefig('tiktok_time_series_decomposition.png', dpi=300)
plt.show()

In [ ]:
print("=== TECHNICAL AUDIT: AUGMENTED DICKEY-FULLER HYPOTHESIS TEST ===")
print("Hypothesis Architecture:")
print(" - H0 (Null Hypothesis): The time series contains a unit root (is non-stationary).")
print(" - Ha (Alternative Hypothesis): The time series does not contain a unit root (is stationary).\n")

adf_result = adfuller(df_resampled['view_count'].dropna())

print(f"ADF Statistic:          {adf_result[0]:.6f}")
print(f"p-value:                {adf_result[1]:.6f}")
print(f"Used Lags:              {adf_result[2]}")
print(f"Number of Observations: {adf_result[3]}")
print("Critical Threshold Values Mapping:")
for key, value in adf_result[4].items():
    print(f"   ├── {key:<5}: {value:.6f}")

p_val = adf_result[1]
if p_val <= 0.05:
    print(f"\nConclusion: p-value ({p_val:.4f}) <= 0.05. Reject H0. The series is STATIONARY.")
    differencing_required = False
else:
    print(f"\nConclusion: p-value ({p_val:.4f}) > 0.05. Fail to reject H0. The series is NON-STATIONARY.")
    differencing_required = True
    print("Action Required: Apply first-order differencing transformations before fitting forecasting models.")

In [ ]:
# Extract mean seasonal adjustments grouped systematically by weekday
df_analysis = df_resampled.copy()
df_analysis['seasonal_signal'] = seasonal_component
df_analysis['day_of_week'] = df_analysis.index.day_name()

weekday_ordering = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_summary = df_analysis.groupby('day_of_week')['seasonal_signal'].mean().reindex(weekday_ordering)

# Plot seasonal impact factors by calendar day
plt.figure(figsize=(9, 4.5))
sns.barplot(x=weekly_summary.index, y=weekly_summary.values, palette='coolwarm', hue=weekly_summary.index, legend=False)
plt.title('TikTok Algorithm Pulse: Weekly Seasonal Impact Breakdown', fontsize=12)
plt.ylabel('Baseline Shift in Total Views')
plt.xlabel('Calendar Schedule Day')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('tiktok_weekly_seasonality_pulse.png', dpi=300)
plt.show()

# Isolate peak delivery window
top_day = weekly_summary.idxmax()
print(f"Optimal Content Distribution Target Isolated: Peak positive algorithm scaling occurs on {top_day}s.")